In [7]:
import os
import json
import time
import random
import itertools
import subprocess
from collections import defaultdict

import pandas as pd
from tqdm.notebook import tqdm

DATA_PATH = "modfied_controlled_agents_qwen.csv"
OUTPUT_DIR = "outputs_qwen"
LOG_FILE = "generation_log_qwen.txt"
MODEL_NAME = "qwen2.5:7b"

pair_types = [(1, 5), (2, 4), (3, 3)]
n_pairs = 5
n_rounds = 3
reflection_round = True

df = pd.read_csv(DATA_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)
agent_usage = defaultdict(int)

def ollama_generate(prompt, model=None, timeout=90):
    if model is None:
        model = MODEL_NAME
    try:
        result = subprocess.run(
            ["ollama", "run", model],
            input=prompt.encode("utf-8"),
            capture_output=True,
            timeout=timeout
        )
        return " ".join(result.stdout.decode("utf-8").strip().split())
    except Exception as e:
        return f"[Error: {e}]"

def log_write(text, log_file=LOG_FILE):
    timestamp = time.strftime("[%Y-%m-%d %H:%M:%S] ")
    with open(log_file, "a", encoding="utf-8") as log:
        log.write(timestamp + text + "\n")

with open(LOG_FILE, "w", encoding="utf-8") as log:
    log.write("=== Debate Log ===\n")

def simulate_dialogue(agent_a, agent_b, topic_statement, model=None):
    topic_id = int(agent_a["topic_id"])
    pref_a = int(agent_a["topic_preference"])
    pref_b = int(agent_b["topic_preference"])

    dialogue = {
        "topic_id": topic_id,
        "pair_type": [pref_a, pref_b],
        "topic_statement": topic_statement,
        "agents": [
            {
                "id": int(agent_a.name),
                "occupation": agent_a["occupation"],
                "region": agent_a["region"],
                "pref": pref_a,
                "gender": agent_a["gender"]
            },
            {
                "id": int(agent_b.name),
                "occupation": agent_b["occupation"],
                "region": agent_b["region"],
                "pref": pref_b,
                "gender": agent_b["gender"]
            }
        ],
        "rounds": []
    }

    for r in range(1, n_rounds + 1):
        prompt_a = (
            f"Topic: {topic_statement}\n"
            f"{agent_a['occupation']} from {agent_a['region']} debates with "
            f"{agent_b['occupation']} from {agent_b['region']}.\n"
            f"Round {r}: {str(agent_a['Preference_Response'])[:400]}"
        )
        prompt_b = (
            f"Topic: {topic_statement}\n"
            f"{agent_b['occupation']} from {agent_b['region']} responds to "
            f"{agent_a['occupation']}.\n"
            f"Round {r}: {str(agent_b['Preference_Response'])[:400]}"
        )

        reply_a = ollama_generate(prompt_a, model=model)
        reply_b = ollama_generate(prompt_b, model=model)

        print(f"[Topic {topic_id}] Pair ({pref_a},{pref_b}) - Round {r} completed")

        dialogue["rounds"].append({
            "round": r,
            "A": reply_a,
            "B": reply_b
        })

    if reflection_round:
        ra = ollama_generate(
            f"As {agent_a['occupation']} from {agent_a['region']}, reflect briefly on '{topic_statement}'.",
            model=model
        )
        rb = ollama_generate(
            f"As {agent_b['occupation']} from {agent_b['region']}, reflect briefly on '{topic_statement}'.",
            model=model
        )
        dialogue["reflection"] = {"A": ra, "B": rb}

        print(f"[Topic {topic_id}] Pair ({pref_a},{pref_b}) - Reflection completed")

    return dialogue

def run_debate_for_pair(topic_id, pair, topic_df, topic_statement, model=None):
    random.seed(int(topic_id) * 100 + int(pair[0]) * 10 + int(pair[1]))
    results = []

    agents_a = topic_df[topic_df["topic_preference"] == pair[0]]
    agents_b = topic_df[topic_df["topic_preference"] == pair[1]]

    if pair == (3, 3):
        all_pairs = list(itertools.combinations(agents_a.index, 2))
    else:
        all_pairs = list(itertools.product(agents_a.index, agents_b.index))

    if not all_pairs:
        return results

    selected_pairs = random.sample(all_pairs, k=min(n_pairs, len(all_pairs)))

    for (idx_a, idx_b) in selected_pairs:
        agent_a = topic_df.loc[idx_a]
        agent_b = topic_df.loc[idx_b]

        if pair == (3, 3) and (agent_a["region"] == agent_b["region"]) and (agent_a["occupation"] == agent_b["occupation"]):
            continue

        agent_usage[idx_a] += 1
        agent_usage[idx_b] += 1

        print(f"[Topic {topic_id}] Starting Pair ({int(agent_a['topic_preference'])},{int(agent_b['topic_preference'])})")
        dialogue = simulate_dialogue(agent_a, agent_b, topic_statement, model=model)
        results.append(dialogue)
        print(f"[Topic {topic_id}] Finished Pair ({int(agent_a['topic_preference'])},{int(agent_b['topic_preference'])})\n")

    return results

def save_results(topic_id, results, output_dir=OUTPUT_DIR):
    path = os.path.join(output_dir, f"topic_{topic_id}.jsonl")
    with open(path, "a", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[Topic {topic_id}] Saved {len(results)} dialogues → {path}")

def process_topic(topic_id, model=None, output_dir=None):
    topic_df = df[df["topic_id"] == topic_id]
    topic_statement = topic_df["statement"].iloc[0]
    topic_results = []

    print(f"\n===== Starting Topic {topic_id} =====")

    for pair in pair_types:
        topic_results.extend(
            run_debate_for_pair(topic_id, pair, topic_df, topic_statement, model=model)
        )

    save_results(topic_id, topic_results, output_dir or OUTPUT_DIR)

    print(f"===== Topic {topic_id} completed ✓ ({len(topic_results)} dialogues) =====\n")
    return topic_id, len(topic_results)

def run_all_topics(topics=None, model=None, output_dir=None, log_file=None):
    if topics is None:
        topics = df["topic_id"].unique()

    if output_dir is None:
        output_dir = OUTPUT_DIR

    # ---- detect completed topics ----
    finished_topics = set()
    for tid in topics:
        path = os.path.join(output_dir, f"topic_{tid}.jsonl")
        if os.path.exists(path) and os.path.getsize(path) > 0:
            finished_topics.add(tid)

    # ---- remaining topics ----
    remaining = [t for t in topics if t not in finished_topics]

    print(f"Total topics: {len(topics)}")
    print(f"Completed topics: {sorted(finished_topics)}")
    print(f"Remaining topics: {remaining}")

    if len(remaining) == 0:
        print("All topics already completed. Nothing to run.")
        return

    log_write(f"Resuming run. {len(remaining)} topics left.", log_file or LOG_FILE)

    for t in tqdm(remaining):
        process_topic(t, model=model, output_dir=output_dir)

    log_write("All topics done", log_file or LOG_FILE)
    print("All topics completed ✓")

In [8]:
run_all_topics()

Total topics: 6
Completed topics: [1, 2, 3, 4, 6]
Remaining topics: [5]


  0%|          | 0/1 [00:00<?, ?it/s]


===== Starting Topic 5 =====
[Topic 5] Starting Pair (1,5)
[Topic 5] Pair (1,5) - Round 1 completed
[Topic 5] Pair (1,5) - Round 2 completed
[Topic 5] Pair (1,5) - Round 3 completed
[Topic 5] Pair (1,5) - Reflection completed
[Topic 5] Finished Pair (1,5)

[Topic 5] Starting Pair (1,5)
[Topic 5] Pair (1,5) - Round 1 completed
[Topic 5] Pair (1,5) - Round 2 completed
[Topic 5] Pair (1,5) - Round 3 completed
[Topic 5] Pair (1,5) - Reflection completed
[Topic 5] Finished Pair (1,5)

[Topic 5] Starting Pair (1,5)
[Topic 5] Pair (1,5) - Round 1 completed
[Topic 5] Pair (1,5) - Round 2 completed
[Topic 5] Pair (1,5) - Round 3 completed
[Topic 5] Pair (1,5) - Reflection completed
[Topic 5] Finished Pair (1,5)

[Topic 5] Starting Pair (1,5)
[Topic 5] Pair (1,5) - Round 1 completed
[Topic 5] Pair (1,5) - Round 2 completed
[Topic 5] Pair (1,5) - Round 3 completed
[Topic 5] Pair (1,5) - Reflection completed
[Topic 5] Finished Pair (1,5)

[Topic 5] Starting Pair (1,5)
[Topic 5] Pair (1,5) - Round

In [9]:
import json
from pathlib import Path

# Folder containing topic JSONL files
base = Path("outputs_qwen")

# Automatically detect topic_*.jsonl
files = sorted(base.glob("topic_*.jsonl"))

print(f"Detected {len(files)} jsonl files:")
for f in files:
    print(" -", f.name)

Detected 6 jsonl files:
 - topic_1.jsonl
 - topic_2.jsonl
 - topic_3.jsonl
 - topic_4.jsonl
 - topic_5.jsonl
 - topic_6.jsonl


In [10]:
rows = []

for file in files:
    print(f"\nProcessing: {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line.strip())

                topic_id = data.get("topic_id")
                pair_type = str(data.get("pair_type"))
                topic_statement = data.get("topic_statement", "")

                agents = data.get("agents", [])
                A = agents[0] if len(agents) > 0 else {}
                B = agents[1] if len(agents) > 1 else {}

                # Extract rounds
                rounds = data.get("rounds", [])
                round_dict = {}
                for i, r in enumerate(rounds, 1):
                    round_dict[f"round{i}_A"] = r.get("A", "")
                    round_dict[f"round{i}_B"] = r.get("B", "")

                # Reflection
                reflection = data.get("reflection", {})
                reflection_A = reflection.get("A", "")
                reflection_B = reflection.get("B", "")

                # Each JSONL → 1 row in CSV
                row = {
                    "topic_id": topic_id,
                    "pair_type": pair_type,
                    "topic_statement": topic_statement,
                    # Agent A meta
                    "A_id": A.get("id", ""),
                    "A_occupation": A.get("occupation", ""),
                    "A_region": A.get("region", ""),
                    "A_pref": A.get("pref", ""),
                    # Agent B meta
                    "B_id": B.get("id", ""),
                    "B_occupation": B.get("occupation", ""),
                    "B_region": B.get("region", ""),
                    "B_pref": B.get("pref", ""),
                    # Rounds
                    **round_dict,
                    # Reflection
                    "reflection_A": reflection_A,
                    "reflection_B": reflection_B,
                }

                rows.append(row)

            except Exception as e:
                print(f"❌ Error parsing {file.name}: {e}")


# Convert to DataFrame & save
df = pd.DataFrame(rows)
output_path = Path("new_debate_results_qwen.csv")

df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ Done! Saved {len(df)} debates to {output_path.resolve()}")


Processing: topic_1.jsonl

Processing: topic_2.jsonl

Processing: topic_3.jsonl

Processing: topic_4.jsonl

Processing: topic_5.jsonl

Processing: topic_6.jsonl

✅ Done! Saved 89 debates to /Users/chennan/PycharmProjects/teamproject/new_debate_results_qwen.csv
